# ToolACE → fin_agent: what the model actually sees

This notebook runs **end-to-end from the live Hugging Face source** — it pulls
`Team-ACE/ToolACE` directly from the Hub and runs it through the actual
`ACEToolDatasetProcessor` (`prepare_dataset.py`'s `extract`/`prepare`/`save` pipeline,
imported, not reimplemented), all inside this notebook. It doesn't read from the
pre-generated `data/*.jsonl` split files — every example below is produced live,
starting from the raw Hub dataset.

The goal is to make concrete two things:

1. What the **system** and **user** messages look like when sent to the LLM.
2. What the model is **expected to produce** in response — shown both in ToolACE's
   native call notation (as the Hub dataset stores it) and in the JSON schema actually
   used as the training target (see `1_training/train_lora.py`'s `render_assistant_turn`).

Regenerating the official train/val/test splits is still `prepare_dataset.py`'s job —
this notebook is a display/walkthrough, not a replacement for it.

In [1]:
import json
import logging
from collections import Counter

logging.getLogger("httpx").setLevel(logging.WARNING)  # quiet HF download noise later on

from prepare_dataset import DATASET_ID, ACEToolDatasetProcessor, try_parse_calls

processor = ACEToolDatasetProcessor()
raw = processor.extract()
print(f"Pulled {len(raw)} raw rows live from https://huggingface.co/datasets/{DATASET_ID}")


/Users/aaron/Projects/fin_agent/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO: Loading Team-ACE/ToolACE ...


INFO: Loaded 11300 raw rows


Pulled 11300 raw rows live from https://huggingface.co/datasets/Team-ACE/ToolACE


## Step 1 — what Hugging Face actually returns (before any processing)

The Hub dataset has exactly two columns: `system` (a string bundling instructions *and*
the tool list as embedded JSON) and `conversations` (a list of `{from, value}` turns, in
ToolACE's own call notation). This is the raw input, untouched.

In [2]:
raw_row = raw[0]
print("Raw HF row keys:", list(raw_row.keys()))
print()
print("raw['system'] (first 600 of", len(raw_row["system"]), "chars):")
print(raw_row["system"][:600], "...")
print()
print("raw['conversations'] (first 2 of", len(raw_row["conversations"]), "turns, exactly as datasets.load_dataset returns them):")
for turn in raw_row["conversations"][:2]:
    print(" ", turn)


Raw HF row keys: ['system', 'conversations']

raw['system'] (first 600 of 4016 chars):
You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out. You should only return the function call in tools call sections.
Here is a list of functions in JSON format that you can invoke:
[{"name": "newAddress", "description": "Generates a new Ethereum address that can be used to send or receive funds. Do not lose t ...

raw['conversations'] (first 2 of 10 turns, exactly as datasets.load_dataset returns them):
  {'from': 'user', 'value': "I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?"}
  {'from': 'assistant', 'value': '[Market Tren

## Step 2 — normalize every row, live

`processor.prepare(raw)` is where every preprocessing fix lives: extracting the embedded
tool schema (or recovering it from ToolACE's alternate `{"tool_name": ...}` template —
see `try_recover_alternate_tool_schema`), normalizing conversation roles, classifying
`call_type`, and dropping rows that can't be safely recovered — with a reason breakdown
logged below (see `0_data/README.md`'s "Known limitation" note for what each reason
means and why the brittle templates are dropped rather than guessed at).

In [3]:
normalized = processor.prepare(raw)
print(f"{len(normalized)} normalized (see the WARNING log line above for the drop-reason breakdown)")


10573 normalized (see the WARNING log line above for the drop-reason breakdown)


## Step 3 — the same row, before and after

Side-by-side: the raw Hub row from Step 1, and what `ACEToolDatasetProcessor.normalize_example` turns it into. It returns `(example, drop_reason)` — `drop_reason` is `None` on success.

In [4]:
normalized_row0, drop_reason = processor.normalize_example(raw_row)
if normalized_row0 is None:
    print(f"(row 0 was dropped — reason: {drop_reason} — showing the first successfully normalized row instead)")
    normalized_row0 = normalized[0]

print(json.dumps(normalized_row0, indent=2)[:1800])


{
  "system": "You are an expert in composing functions. You are given a question and a set of possible functions. \nBased on the question, you will need to make one or more function/tool calls to achieve the purpose. \nIf none of the function can be used, point it out. If the given question lacks the parameters required by the function,\nalso point it out. You should only return the function call in tools call sections.\nHere is a list of functions in JSON format that you can invoke:\n[{\"name\": \"newAddress\", \"description\": \"Generates a new Ethereum address that can be used to send or receive funds. Do not lose the password! We can't restore access to an address if you lose it.\", \"parameters\": {\"type\": \"dict\", \"properties\": {\"password\": {\"description\": \"The password for the new Ethereum address\", \"type\": \"string\"}}, \"required\": [\"password\"]}, \"required\": null}, {\"name\": \"Market Trends API\", \"description\": \"Get the latest market trends and relevant

## Rendering the actual training target

Every assistant turn is normalized in ToolACE's own notation, e.g.
`Get User's Likes(user_id="12345", limit=5)`. That is **not** what the model is trained
to produce — `1_training/train_lora.py` re-renders call turns into the JSON schema the
production agentic graph's validator expects. `render_assistant_turn` below is the exact
same function (reproduced here rather than imported, since `train_lora.py` also imports
`peft`/`trl`/`transformers.AutoModelForCausalLM`, which this notebook doesn't need just
to demonstrate formatting).

In [5]:
def render_assistant_turn(content: str) -> str:
    """Mirrors 1_training/train_lora.py's render_assistant_turn."""
    calls = try_parse_calls(content)
    if calls is None:
        return content
    return json.dumps(
        [{"name": c["name"], "arguments": c["arguments"]} for c in calls], indent=2
    )


In [6]:
def show_example(example, max_tools_shown=2, max_chars=400):
    preamble_end = example["system"].find("[")
    preamble = example["system"][:preamble_end].strip()
    print("=" * 88)
    print("SYSTEM MESSAGE — instructions (sent once, at the start of every conversation)")
    print("=" * 88)
    print(preamble)

    n_tools = len(example["tools"])
    print()
    print("=" * 88)
    print(f"SYSTEM MESSAGE — available tools ({n_tools} total, showing {min(max_tools_shown, n_tools)})")
    print("=" * 88)
    for tool in example["tools"][:max_tools_shown]:
        print(json.dumps(tool, indent=2))
    if n_tools > max_tools_shown:
        remaining = [t["name"] for t in example["tools"][max_tools_shown:]]
        print(f"... and {n_tools - max_tools_shown} more tool(s): {remaining}")

    for i, turn in enumerate(example["turns"]):
        role = turn["role"].upper()
        content = turn["content"]
        if turn["role"] == "assistant":
            calls = try_parse_calls(content)
            print()
            print("-" * 88)
            if calls is not None:
                print(f"[{i}] ASSISTANT  <-- MODEL IS EXPECTED TO GENERATE THIS (a function call)")
                print("-" * 88)
                print("As stored in the dataset (ToolACE native syntax):")
                print(" ", content)
                print()
                print("As used for the actual training target (JSON — matches production's validator):")
                print(" ", render_assistant_turn(content))
            else:
                print(f"[{i}] ASSISTANT  <-- MODEL IS EXPECTED TO GENERATE THIS (natural-language reply, no call)")
                print("-" * 88)
                print(" ", content[:max_chars] + ("..." if len(content) > max_chars else ""))
        else:
            print()
            print("-" * 88)
            print(f"[{i}] {role}  (input context — not generated by the model)")
            print("-" * 88)
            print(" ", content[:max_chars] + ("..." if len(content) > max_chars else ""))


## Example 1 — single function call

One user request, one function call expected in response.

In [7]:
single_example = next(e for e in normalized if e["call_type"] == "single")
show_example(single_example)


SYSTEM MESSAGE — instructions (sent once, at the start of every conversation)
You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out. You should only return the function call in tools call sections.
Here is a list of functions in JSON format that you can invoke:

SYSTEM MESSAGE — available tools (3 total, showing 2)
{
  "name": "Trending Videos",
  "description": "Retrieves a list of trending videos from YouTube, filtered by locale, country, and type.",
  "parameters": {
    "type": "dict",
    "properties": {
      "hl": {
        "description": "Locale/language for the request",
        "type": "string",
        "default": "en"
      },
      "gl": {
        "description": "Country to get trending video

## Example 2 — parallel function calls

The user's request requires more than one function call *in the same turn* — the model has to emit a list of calls, not just one.

In [8]:
parallel_example = next(e for e in normalized if e["call_type"] == "parallel")
show_example(parallel_example)


SYSTEM MESSAGE — instructions (sent once, at the start of every conversation)
You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out. You should only return the function call in tools call sections.
Here is a list of functions in JSON format that you can invoke:

SYSTEM MESSAGE — available tools (5 total, showing 2)
{
  "name": "Search Flights by Location",
  "description": "Search for airports and locations by name and return a list of matching results",
  "parameters": {
    "type": "dict",
    "properties": {
      "name": {
        "description": "The name of the airport or location to search for",
        "type": "string"
      }
    },
    "required": [
      "name"
    ]
  },
  "required": null
}
{

## Example 3 — multi-turn tool use

The model is asked to generate output at **more than one point** in the same conversation — e.g. call a function, receive the tool result, then either call another function or answer using that result. Every `ASSISTANT` block below is a separate point where the model has to produce something.

In [9]:
multi_turn_example = next(e for e in normalized if e["call_type"] == "multi_turn")
show_example(multi_turn_example)


SYSTEM MESSAGE — instructions (sent once, at the start of every conversation)
You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out. You should only return the function call in tools call sections.
Here is a list of functions in JSON format that you can invoke:

SYSTEM MESSAGE — available tools (6 total, showing 2)
{
  "name": "newAddress",
  "description": "Generates a new Ethereum address that can be used to send or receive funds. Do not lose the password! We can't restore access to an address if you lose it.",
  "parameters": {
    "type": "dict",
    "properties": {
      "password": {
        "description": "The password for the new Ethereum address",
        "type": "string"
      }
    },
    "req

## Example 4 — no function call is appropriate

Not every user turn should trigger a call — sometimes the correct response is a direct answer, or a request for clarification. `try_parse_calls` returns `None` for these, and training/eval both treat that as the target (see `run_internal_eval.py`'s refusal-accuracy metric).

In [10]:
no_call_example = next(e for e in normalized if e["call_type"] == "no_call")
show_example(no_call_example)


SYSTEM MESSAGE — instructions (sent once, at the start of every conversation)
You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out. You should only return the function call in tools call sections.
Here is a list of functions in JSON format that you can invoke:

SYSTEM MESSAGE — available tools (6 total, showing 2)
{
  "name": "getESGScores",
  "description": "This API provides real-time Environmental, Social, Governance and Overall scores for companies on a scale of 0 to 100. In addition to this, the API also provides other relevant metrics like Global Rank, Industry Rank and more.",
  "parameters": {
    "type": "dict",
    "properties": {
      "isin": {
        "description": "International Securitie

## What the model literally receives: the chat template

The message list above is the *logical* view. In practice, the base model's tokenizer
flattens `[{"role": ..., "content": ...}, ...]` into a single string via its chat
template before anything reaches the model — this is what `train_lora.py` calls
`tokenizer.apply_chat_template` for.

Below, everything up to (but not including) the final assistant turn is the **input**
the model is conditioned on; the final assistant turn's content is the **target** it's
trained to produce. `enable_thinking=False` matches the non-thinking mode
`2_evaluations/` uses everywhere else — Qwen3 is a hybrid reasoning model that can emit
`<think>...</think>` before answering, which production has no latency budget for. This
only downloads the tokenizer's small config/vocab files, not any model weights.

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

messages = [{"role": "system", "content": single_example["system"]}]
for turn in single_example["turns"]:
    content = turn["content"]
    if turn["role"] == "assistant":
        content = render_assistant_turn(content)
    messages.append({"role": turn["role"], "content": content})

prompt_messages = messages[:-1]
target = messages[-1]["content"]

rendered_prompt = tokenizer.apply_chat_template(
    prompt_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
print(rendered_prompt)
print()
print("=" * 30, "MODEL IS TRAINED TO GENERATE", "=" * 30)
print(target)


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


<|im_start|>system
You are an expert in composing functions. You are given a question and a set of possible functions. 
Based on the question, you will need to make one or more function/tool calls to achieve the purpose. 
If none of the function can be used, point it out. If the given question lacks the parameters required by the function,
also point it out. You should only return the function call in tools call sections.
Here is a list of functions in JSON format that you can invoke:
[{"name": "Trending Videos", "description": "Retrieves a list of trending videos from YouTube, filtered by locale, country, and type.", "parameters": {"type": "dict", "properties": {"hl": {"description": "Locale/language for the request", "type": "string", "default": "en"}, "gl": {"description": "Country to get trending videos from", "type": "string", "default": "US"}, "type": {"description": "Type of trending videos", "type": "string", "default": "mu"}}, "required": []}, "required": null}, {"name": "Get 

How long is that, in tokens? This grounds the ~4K-tokens/request assumption used in `4_deployment/`'s memory-budget table against a real example rather than a guess.

In [12]:
prompt_tokens = len(tokenizer(rendered_prompt)["input_ids"])
target_tokens = len(tokenizer(target)["input_ids"])
print(f"Prompt: {prompt_tokens} tokens")
print(f"Target: {target_tokens} tokens")
print(f"Total:  {prompt_tokens + target_tokens} tokens")


Prompt: 1537 tokens
Target: 20 tokens
Total:  1557 tokens


## Dataset-wide summary

The call-type mix across everything just pulled and normalized live from the Hub (the full 11,300-row `train` split, before any train/val/test split — that split is `prepare_dataset.py`'s job, not this notebook's).

In [13]:
counts = Counter(e["call_type"] for e in normalized)
total = len(normalized)
for call_type, n in sorted(counts.items()):
    print(f"{call_type:12s} {n:5d}  ({n/total:.1%})")


multi_turn     650  (6.1%)
no_call       1338  (12.7%)
parallel      4045  (38.3%)
single        4540  (42.9%)
